# Chapter 3 — Information Theory for Machine Learning

**Math for ML Engineers** · companion notebook

This notebook mirrors §3.1–§3.4 of the chapter (entropy, cross-entropy, KL divergence, and mutual information), building each quantity from its definition and then verifying the identities that make them useful in practice. **Key takeaway: KL divergence is not a niche information-theory curiosity — it is the single most-reused quantity in this book, silently underlying cross-entropy loss, the VAE/ELBO objective, and the KL penalty inside DPO/RLHF.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Setup and derivations

Entropy measures the average "surprise" of samples drawn from a distribution $p$ (§3.1):

$$H(p) = -\sum_i p_i \log p_i$$

Cross-entropy measures the average surprise of samples from $p$ when we *score* them using a different distribution $q$ (§3.2):

$$H(p, q) = -\sum_i p_i \log q_i$$

The Kullback–Leibler divergence is the gap between the two — the extra bits/nats you pay for using $q$ instead of the true $p$ (§3.2):

$$\mathrm{KL}(p \,\|\, q) = \sum_i p_i \log \frac{p_i}{q_i}$$

These three are linked by one identity we will verify numerically below:

$$H(p, q) = H(p) + \mathrm{KL}(p \,\|\, q)$$

Gibbs' inequality guarantees $\mathrm{KL}(p\,\|\,q) \ge 0$, with equality iff $p = q$ almost everywhere — we will also confirm this empirically across many random distribution pairs. Finally, mutual information (§3.3) measures shared information between two variables:

$$I(X;Y) = \mathrm{KL}\big(p(x,y) \,\|\, p(x)p(y)\big) = H(X) + H(Y) - H(X,Y)$$

In [ ]:
def entropy(p, eps=1e-12):
    """H(p) = -Σ pᵢ log pᵢ  (nats, natural log)."""
    p = np.asarray(p, dtype=float)
    p = p / p.sum()
    return -np.sum(p * np.log(p + eps))


def cross_entropy(p, q, eps=1e-12):
    """H(p, q) = -Σ pᵢ log qᵢ."""
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    p = p / p.sum(); q = q / q.sum()
    return -np.sum(p * np.log(q + eps))


def kl_divergence(p, q, eps=1e-12):
    """KL(p || q) = Σ pᵢ log(pᵢ/qᵢ). Always ≥ 0."""
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    p = p / p.sum(); q = q / q.sum()
    return np.sum(p * np.log((p + eps) / (q + eps)))


def mutual_information(joint_pxy):
    """I(X;Y) = KL(p(x,y) || p(x)p(y)) = H(X) + H(Y) - H(X,Y)."""
    pxy = joint_pxy / joint_pxy.sum()
    px  = pxy.sum(axis=1)
    py  = pxy.sum(axis=0)
    return entropy(px) + entropy(py) - entropy(pxy.ravel())


def gaussian_kl(mu1, sigma1, mu2, sigma2):
    """KL(N(μ1,σ1²) || N(μ2,σ2²)) closed form:
    KL = log(σ2/σ1) + (σ1² + (μ1-μ2)²)/(2σ2²) - 1/2"""
    return (np.log(sigma2 / sigma1)
            + (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2)
            - 0.5)

rng = np.random.default_rng(3)
print("Functions defined: entropy, cross_entropy, kl_divergence, mutual_information, gaussian_kl")

## Verifying the identity and non-negativity

Two things need checking before we trust these functions in a training loop:

1. **The decomposition** $H(p,q) = H(p) + \mathrm{KL}(p\|q)$ should hold to floating-point precision for *any* pair of categorical distributions, not just convenient toy examples.
2. **Non-negativity** of KL should hold for *every* pair, including wildly mismatched ones (e.g. $p$ nearly one-hot, $q$ nearly uniform).

We test both by drawing many random categorical distributions (via Dirichlet sampling, which naturally produces valid probability vectors) and checking the identity and sign on each pair.

In [ ]:
n_trials = 2000
n_categories = 6

kl_values = np.empty(n_trials)
identity_gap = np.empty(n_trials)

for i in range(n_trials):
    # Dirichlet(alpha small) gives spiky, varied distributions -> good stress test
    alpha = rng.uniform(0.3, 3.0)
    p = rng.dirichlet(alpha * np.ones(n_categories))
    q = rng.dirichlet(alpha * np.ones(n_categories))

    kl_pq = kl_divergence(p, q)
    kl_values[i] = kl_pq

    lhs = cross_entropy(p, q)
    rhs = entropy(p) + kl_pq
    identity_gap[i] = abs(lhs - rhs)

print(f"H(p,q) = H(p) + KL(p||q) identity: max abs gap over {n_trials} trials = {identity_gap.max():.3e}")
print(f"KL non-negativity: min KL value over {n_trials} trials = {kl_values.min():.3e}")
print(f"KL non-negativity: fraction of trials with KL < 0: {(kl_values < 0).mean():.4%}")

## Visualizing the identity and the KL sweep

Two plots make these results concrete:

- **Left:** a histogram of the ~2000 sampled $\mathrm{KL}(p\|q)$ values, confirming the entire mass sits at or above zero — Gibbs' inequality holding empirically, not just in the mean.
- **Right:** we fix a target distribution $p$ over 5 categories and construct a family of $q_t = (1-t)\,p + t\,u$ that interpolates from a mismatched distribution $u$ toward the true $p$ as $t \to 0$. Plotting $\mathrm{KL}(p\|q_t)$ against $t$ shows the divergence shrinking smoothly to zero exactly as $q_t \to p$ — visually, this is what "the model is learning" looks like from an information-theoretic view.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: histogram of KL values across random pairs
axes[0].hist(kl_values, bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="crimson", linestyle="--", linewidth=1.5, label="KL = 0")
axes[0].set_title(f"KL(p‖q) over {n_trials} random pairs\n(all ≥ 0, min = {kl_values.min():.2e})")
axes[0].set_xlabel("KL divergence (nats)")
axes[0].set_ylabel("count")
axes[0].legend()

# Right: sweep q toward p, watch KL -> 0
p_target = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
u_mismatched = np.array([0.05, 0.1, 0.15, 0.2, 0.5])  # reversed / mismatched

ts = np.linspace(1.0, 0.0, 100)
kl_sweep = np.array([kl_divergence(p_target, (1 - t) * p_target + t * u_mismatched) for t in ts])

axes[1].plot(ts, kl_sweep, color="darkorange", linewidth=2)
axes[1].scatter([1.0, 0.0], [kl_sweep[0], kl_sweep[-1]], color=["crimson", "seagreen"], zorder=5)
axes[1].annotate(f"q=u, KL={kl_sweep[0]:.3f}", (1.0, kl_sweep[0]),
                  textcoords="offset points", xytext=(-10, 12), ha="right", color="crimson")
axes[1].annotate(f"q=p, KL={kl_sweep[-1]:.3f}", (0.0, kl_sweep[-1]),
                  textcoords="offset points", xytext=(10, 12), color="seagreen")
axes[1].set_title("KL(p‖q_t) as q_t → p")
axes[1].set_xlabel("t  (t=1: q far from p, t=0: q=p)")
axes[1].set_ylabel("KL divergence (nats)")
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

print(f"KL at t=1.0 (q maximally mismatched): {kl_sweep[0]:.4f} nats")
print(f"KL at t=0.0 (q == p exactly):        {kl_sweep[-1]:.4e} nats")
print("As predicted by Gibbs' inequality, KL only reaches its floor of 0 when q matches p exactly.")

## Why this matters in production: KL is the objective, not just a diagnostic

This is the most important idea in the chapter, and it is easy to miss because KL divergence is usually introduced as a "distance-like" diagnostic rather than as the thing you are actually optimizing. In practice it is almost always the *actual* training objective, wearing a different name:

- **Cross-entropy loss *is* KL minimization.** For classification with one-hot true labels $p$, $H(p)$ is a constant (it's exactly $0$ for a one-hot vector, since all the probability mass sits on one outcome). Because $H(p,q) = H(p) + \mathrm{KL}(p\|q)$ and $H(p)$ doesn't depend on the model's parameters, minimizing cross-entropy loss $H(p,q)$ over model outputs $q$ is *identical* to minimizing $\mathrm{KL}(p\|q)$ — the standard softmax + cross-entropy training loop used everywhere from MNIST classifiers to LLM next-token prediction is KL minimization in disguise.
- **`gaussian_kl` is the ELBO's regularizer.** In a VAE, the loss is a reconstruction term plus $\mathrm{KL}(q_\phi(z|x)\,\|\,p(z))$, the exact closed-form Gaussian KL implemented above, pulling the learned latent posterior toward the prior (usually $\mathcal{N}(0, I)$).
- **DPO's implicit reward *is* a KL penalty.** Direct Preference Optimization (and RLHF more broadly) optimizes a policy $\pi_\theta$ against a reference policy $\pi_{\mathrm{ref}}$ with an explicit $\beta \cdot \mathrm{KL}(\pi_\theta \,\|\, \pi_{\mathrm{ref}})$ penalty term controlling how far the fine-tuned model is allowed to drift from its pretrained/SFT behavior — too small a $\beta$ and the model reward-hacks; too large and it never moves.

Zoom out and nearly every loss function in this book — softmax cross-entropy, the VAE's ELBO, and the RLHF/DPO objective — is some form of $\mathrm{KL}(p\|q)$ or a $\beta$-scaled variant of it. **If you understand `kl_divergence` and `gaussian_kl` in this notebook, you already understand the mathematical core of supervised classification, variational autoencoders, and LLM alignment.**

In [ ]:
# Experiment 1: cross-entropy loss minimization IS KL minimization for one-hot labels.
num_classes = 4
p_one_hot = np.zeros(num_classes); p_one_hot[2] = 1.0  # true class = index 2

print("H(p) for one-hot label:", entropy(p_one_hot), "(constant, ~0 regardless of model)")
print()
print(f"{'q (model softmax output)':35s} {'CrossEnt H(p,q)':>18s} {'KL(p||q)':>12s} {'H(p)+KL':>10s}")
for q in [
    np.array([0.05, 0.05, 0.85, 0.05]),   # confident & correct
    np.array([0.25, 0.25, 0.25, 0.25]),   # uniform / untrained
    np.array([0.4,  0.3,  0.1,  0.2 ]),   # confident & WRONG
]:
    ce = cross_entropy(p_one_hot, q)
    kl = kl_divergence(p_one_hot, q)
    print(f"{str(np.round(q, 2)):35s} {ce:18.4f} {kl:12.4f} {entropy(p_one_hot) + kl:10.4f}")
print("\n-> Cross-entropy and KL(p||q) track each other exactly (they differ only by the constant H(p)=0),")
print("   confirming that minimizing softmax cross-entropy loss minimizes KL(true||predicted).")

# Experiment 2: gaussian_kl as the DPO / RLHF-style beta-scaled KL penalty.
print("\n--- Gaussian KL as a policy-drift penalty (VAE prior term / DPO reference penalty) ---")
mu_ref, sigma_ref = 0.0, 1.0          # reference / prior policy N(0,1)
betas = [0.01, 0.1, 1.0]
for mu_policy in [0.0, 0.5, 2.0]:
    kl_drift = gaussian_kl(mu_policy, 1.0, mu_ref, sigma_ref)
    penalties = [beta * kl_drift for beta in betas]
    print(f"policy mean={mu_policy:>4}: KL(policy||ref)={kl_drift:7.4f}  "
          f"beta-scaled penalties {betas} -> {[f'{p:.4f}' for p in penalties]}")
print("\n-> As the fine-tuned policy drifts further from the reference (mu increases),")
print("   the KL penalty grows and, scaled by beta, directly increases the DPO/RLHF loss —")
print("   exactly how beta constrains alignment fine-tuning from moving too far off-distribution.")

## Exercise

1. Modify the `p_one_hot` / cross-entropy experiment above to use a **smoothed label** (e.g. `p = [0.02, 0.02, 0.94, 0.02]` instead of a strict one-hot) as used in label-smoothing regularization. Show that `H(p)` is no longer exactly `0`, and re-derive what cross-entropy loss is now minimizing in terms of `KL(p||q)`. Check whether your logging code reports the smoothed or the unsmoothed cross-entropy before assuming "loss" means what you expect.
2. In the `gaussian_kl` DPO-penalty experiment, hold `mu_policy` fixed and instead sweep `sigma_policy` far below `sigma_ref` (e.g. `sigma_policy = 0.05`) — a policy that has collapsed to near-deterministic outputs. Plot `KL(policy||ref)` against `sigma_policy` and explain, referencing Failure 7 (reward hacking) in Appendix E, why an unconstrained (small or zero) `beta` combined with this kind of variance collapse is a classic RLHF/DPO failure mode rather than a training success.